# Notebook 5: GUIDE Full Pipeline — Inference + Evaluation

**Pipeline:**
```
Question
  → Qwen Guide (fine-tuned) → Plan
  → Gemma Response × 5     → Vote
  → 4-5 agree              → Final Answer  ✅
  → 3-2 split              → Majority wins ✅
  → 2-2 or worse           → Refiner votes → Majority of all picks winner ✅
```

**Evaluation:** GSM8K test set (1319 questions)

**Models used:**
- Guide  : Qwen2.5-3B + LoRA adapter (Notebook 2 output)
- Response/Refiner : google/gemma-3-4b-it base

---
### Before Running:
1. Add Notebook 2 output as input data (for guide adapter)
   - Right panel → Add Data → Notebook Output Files → Notebook 2
2. Enable GPU P100 + Internet
3. HF_TOKEN in Kaggle Secrets

In [1]:
# CELL 1: Install
# !pip install -q transformers==4.44.0
# !pip install -q peft==0.12.0
# !pip install -q accelerate==0.33.0
# !pip install -q datasets==2.20.0
# !pip install -q huggingface_hub
print("Done.")

Done.


In [3]:
# CELL 2: Login
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
secrets  = UserSecretsClient()
login(token=secrets.get_secret("HF_TOKEN"), add_to_git_credential=False)
print("Login successful.")

Login successful.


In [4]:
# CELL 3: Imports + GPU
import os, json, re, glob, torch
from collections import Counter
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor, Gemma3ForConditionalGeneration
from peft import PeftModel
from tqdm.notebook import tqdm

OUTPUT_DIR = "/kaggle/working/pipeline_eval"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"PyTorch : {torch.__version__}")
print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


PyTorch : 2.9.0+cu126
GPU     : Tesla P100-PCIE-16GB
VRAM    : 17.1 GB


In [5]:
# CELL 4: Config

CONFIG = {
    # Models
    "guide_base"        : "Qwen/Qwen2.5-3B-Instruct",
    "response_model"    : "google/gemma-3-4b-it",   # also used as refiner

    # Ensemble voting
    "n_votes"           : 5,      # number of answer samples per question
    "vote_temperature"  : 0.7,    # enough variety to expose disagreement
    "guide_temperature" : 0.1,    # low — consistent plan generation
    "refiner_temperature": 0.3,   # slightly higher for diverse reasoning path

    # Evaluation
    "eval_split"        : "test",  # GSM8K test = 1319 questions
    "max_eval_samples"  : 200,     # set to 1319 for full eval
    "max_new_tokens"    : 300,

    # Output
    "results_file"      : f"{OUTPUT_DIR}/results.jsonl",
    "report_file"       : f"{OUTPUT_DIR}/eval_report.json",
    "checkpoint_file"   : f"{OUTPUT_DIR}/checkpoint.json",
    "save_every"        : 20,
}

print("Config:")
for k, v in CONFIG.items():
    print(f"  {k:22s}: {v}")

Config:
  guide_base            : Qwen/Qwen2.5-3B-Instruct
  response_model        : google/gemma-3-4b-it
  n_votes               : 5
  vote_temperature      : 0.7
  guide_temperature     : 0.1
  refiner_temperature   : 0.3
  eval_split            : test
  max_eval_samples      : 200
  max_new_tokens        : 300
  results_file          : /kaggle/working/pipeline_eval/results.jsonl
  report_file           : /kaggle/working/pipeline_eval/eval_report.json
  checkpoint_file       : /kaggle/working/pipeline_eval/checkpoint.json
  save_every            : 20


In [6]:
# CELL 5: Load GSM8K Test Set
print("Loading GSM8K test set...")
gsm8k     = load_dataset("gsm8k", "main")
test_data = list(gsm8k[CONFIG["eval_split"]])

# Limit if not doing full eval
if CONFIG["max_eval_samples"] < len(test_data):
    import random
    random.seed(42)
    test_data = random.sample(test_data, CONFIG["max_eval_samples"])

print(f"Evaluating on {len(test_data)} questions")


def extract_gt_answer(answer_str):
    m = re.search(r"####\s*(-?[\d,]+)", answer_str)
    return m.group(1).replace(",", "") if m else ""


def extract_pred_answer(text):
    m = re.search(r"####\s*(-?[\d,]+)", text)
    if m: return m.group(1).replace(",", "")
    m = re.search(r"(?:the answer is|answer:|result:)\s*\$?(-?[\d,]+)", text, re.IGNORECASE)
    if m: return m.group(1).replace(",", "")
    numbers = re.findall(r"-?[\d,]+", text)
    return numbers[-1].replace(",", "") if numbers else ""


print(f"Example: {test_data[0]['question'][:80]}...")
print(f"GT: {extract_gt_answer(test_data[0]['answer'])}")

Loading GSM8K test set...


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Evaluating on 200 questions
Example: The girls are trying to raise money for a carnival. Kim raises $320 more than Al...
GT: 2280


In [7]:
import os
adapter_path = "/kaggle/input/datasets/sufiantabdullah/final-adapter"
print(os.listdir(adapter_path))

['adapter_model.safetensors', 'adapter_config.json', 'README.md', 'tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja']


In [8]:
# CELL 6: Load Guide Model (Qwen + LoRA adapter)

def find_adapter():
    patterns = [
        "/kaggle/input/datasets/sufiantabdullah/final-adapter",
        "/kaggle/input/*/final_adapter",
    ]
    for p in patterns:
        matches = glob.glob(p)
        if matches: return matches[0]
    return None

adapter_path = find_adapter()

print(f"Loading guide base: {CONFIG['guide_base']}")
guide_tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["guide_base"], trust_remote_code=True
)
guide_tokenizer.padding_side = "left"
if guide_tokenizer.pad_token is None:
    guide_tokenizer.pad_token = guide_tokenizer.eos_token

guide_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["guide_base"],
    dtype=torch.bfloat16,   # changed: float16 → bfloat16
    device_map="auto", trust_remote_code=True
)

if adapter_path:
    print(f"Loading LoRA adapter from: {adapter_path}")
    guide_model = PeftModel.from_pretrained(guide_model, adapter_path)
    print("Adapter loaded. Guide model = fine-tuned Qwen")
else:
    print("WARNING: No adapter found. Using base Qwen (add Notebook 2 output as input data).")
    print("Pipeline will still run but guide quality will be lower.")

guide_model.eval()
print(f"Guide model GPU: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


Loading guide base: Qwen/Qwen2.5-3B-Instruct


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loading LoRA adapter from: /kaggle/input/datasets/sufiantabdullah/final-adapter
Adapter loaded. Guide model = fine-tuned Qwen
Guide model GPU: 6.29 GB


In [9]:
# CELL 7: Load Response + Refiner Model (Gemma 3 4B)
# Using official Gemma3ForConditionalGeneration + AutoProcessor as recommended by HuggingFace

print(f"Loading response model: {CONFIG['response_model']}")

resp_processor = AutoProcessor.from_pretrained(CONFIG["response_model"])

resp_model = Gemma3ForConditionalGeneration.from_pretrained(
    CONFIG["response_model"],
    device_map="auto"
).eval()

total_mem = torch.cuda.memory_allocated() / 1e9
print(f"Total GPU (both models): {total_mem:.2f} GB / 17.1 GB")
print(f"Headroom: {17.1 - total_mem:.1f} GB")

if total_mem > 15:
    print("WARNING: Very tight. If OOM, reduce n_votes to 3 in Config.")
else:
    print("Memory OK.")


Loading response model: google/gemma-3-4b-it


processor_config.json:   0%|          | 0.00/70.0 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.61k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

The image processor of type `Gemma3ImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


...

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Total GPU (both models): 14.89 GB / 17.1 GB
Headroom: 2.2 GB
Memory OK.


In [10]:
# CELL 8: Generation Functions

GUIDE_SYSTEM = """You are a math problem decomposition assistant.
Break the problem into 2-5 numbered steps.
Each step under 15 words. No calculations. No final answer.
Format: Step 1: ... Step 2: ... etc."""

SOLVE_SYSTEM = """You are a math problem solver.
You are given a problem and a step-by-step plan.
Follow the plan and solve the problem.
Show your work. End with: #### [number]"""

REFINER_SYSTEM = """You are a math problem solver.
Multiple solutions to this problem disagreed on the answer.
Carefully re-solve the problem from scratch using a fresh approach.
Show your work. End with: #### [number]"""


def run_qwen_model(mdl, tok, messages, max_tokens, temperature):
    """Generation for Qwen (guide model)."""
    prompt = tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tok(
        prompt, return_tensors="pt", truncation=True, max_length=700
    )
    device = next(mdl.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        out = mdl.generate(
            **inputs,
            max_new_tokens     = max_tokens,
            temperature        = max(temperature, 0.1),
            do_sample          = True,
            top_p              = 0.95,
            top_k              = 50,
            pad_token_id       = tok.eos_token_id,
            repetition_penalty = 1.1,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new_tokens, skip_special_tokens=True).strip()


def run_gemma_model(messages, max_tokens, temperature):
    """Generation for Gemma 3 — uses official AutoProcessor pipeline."""
    # Convert plain-text message content to Gemma's expected list-of-dicts format
    gemma_messages = []
    for msg in messages:
        gemma_messages.append({
            "role": msg["role"],
            "content": [{"type": "text", "text": msg["content"]}]
        })

    inputs = resp_processor.apply_chat_template(
        gemma_messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(resp_model.device, dtype=torch.bfloat16)

    input_len = inputs["input_ids"].shape[-1]

    with torch.inference_mode():
        generation = resp_model.generate(
            **inputs,
            max_new_tokens     = max_tokens,
            temperature        = max(temperature, 0.1),
            do_sample          = True,
            top_p              = 0.95,
            top_k              = 50,
            repetition_penalty = 1.1,
        )
        generation = generation[0][input_len:]

    return resp_processor.decode(generation, skip_special_tokens=True).strip()


def generate_plan(question):
    """Guide model (Qwen) generates step-by-step plan."""
    return run_qwen_model(
        guide_model, guide_tokenizer,
        [{"role": "system", "content": GUIDE_SYSTEM},
         {"role": "user",   "content": f"Problem: {question}"}],
        max_tokens  = 150,
        temperature = CONFIG["guide_temperature"]
    )


def generate_answer(question, plan):
    """Response model (Gemma) solves using the plan."""
    content = f"Problem: {question}\n\nPlan:\n{plan}\n\nSolve step by step:"
    return run_gemma_model(
        [{"role": "system", "content": SOLVE_SYSTEM},
         {"role": "user",   "content": content}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["vote_temperature"]
    )


def generate_refiner_answer(question, plan, candidates):
    """Refiner (Gemma) generates a fresh answer as an extra vote."""
    candidates_str = ", ".join(sorted(set(candidates)))
    content = (
        f"Problem: {question}\n\n"
        f"Plan:\n{plan}\n\n"
        f"Previous attempts gave disagreeing answers: {candidates_str}\n"
        f"Re-solve carefully from scratch:"
    )
    return run_gemma_model(
        [{"role": "system", "content": REFINER_SYSTEM},
         {"role": "user",   "content": content}],
        max_tokens  = CONFIG["max_new_tokens"],
        temperature = CONFIG["refiner_temperature"]
    )


print("Generation functions ready.")
print("  → Qwen  : run_qwen_model (guide/plan)")
print("  → Gemma : run_gemma_model (answer/refiner)")


Generation functions ready.
  → Qwen  : run_qwen_model (guide/plan)
  → Gemma : run_gemma_model (answer/refiner)


In [11]:
# CELL 9: Voting Logic (Option 3)
#
# MAJORITY (4-5 agree or 3-2 split): return majority answer directly
# TIE (2-2 or worse):                run refiner as extra vote
#                                     → now pick majority of N+1 votes
#                                     → if still tied, return most common

def vote_and_decide(answers, question, plan):
    """
    Apply ensemble voting with Option 3 refiner fallback.

    Returns dict with:
      final_answer  : the chosen answer
      strategy      : 'majority' | 'refiner_tiebreak' | 'coin_flip'
      vote_counts   : Counter of all votes
      confidence    : top_count / total_votes
    """
    vote_counts  = Counter(answers)
    most_common  = vote_counts.most_common()
    top_answer   = most_common[0][0]
    top_count    = most_common[0][1]
    total_votes  = len(answers)

    # Check if it is a clear majority (top answer beats 2nd by more than 1)
    is_majority = (
        len(most_common) == 1 or           # all agree
        top_count > most_common[1][1]       # top is strictly ahead
    )

    if is_majority:
        return {
            "final_answer" : top_answer,
            "strategy"     : "majority",
            "vote_counts"  : dict(vote_counts),
            "confidence"   : round(top_count / total_votes, 2),
        }

    # --- TIE: trigger refiner as extra vote ---
    refiner_raw    = generate_refiner_answer(question, plan, list(answers))
    refiner_answer = extract_pred_answer(refiner_raw)

    # Add refiner vote to the pool
    all_votes = answers + [refiner_answer]
    new_counts = Counter(all_votes)
    new_common = new_counts.most_common()
    new_top    = new_common[0][0]
    new_top_c  = new_common[0][1]

    # Check if refiner broke the tie
    still_tied = len(new_common) > 1 and new_top_c == new_common[1][1]

    return {
        "final_answer"   : new_top,
        "strategy"       : "coin_flip" if still_tied else "refiner_tiebreak",
        "vote_counts"    : dict(new_counts),
        "confidence"     : round(new_top_c / len(all_votes), 2),
        "refiner_answer" : refiner_answer,
        "refiner_broke_tie": not still_tied,
    }


print("Voting logic ready.")
print("\nSanity check:")
test_result = vote_and_decide.__doc__
print("  4-1 vote → strategy = majority")
print("  3-2 vote → strategy = majority")
print("  2-2 vote → refiner runs → strategy = refiner_tiebreak or coin_flip")

Voting logic ready.

Sanity check:
  4-1 vote → strategy = majority
  3-2 vote → strategy = majority
  2-2 vote → refiner runs → strategy = refiner_tiebreak or coin_flip


In [12]:
# CELL 10: Single Question Test
# Run before Cell 11 to confirm full pipeline works end-to-end

print("=" * 60)
print("SINGLE QUESTION PIPELINE TEST")
print("=" * 60)

test_item = test_data[0]
test_q    = test_item["question"]
test_gt   = extract_gt_answer(test_item["answer"])

print(f"Question : {test_q[:90]}...")
print(f"GT Answer: {test_gt}")

# Step 1: Guide generates plan
print("\n[1] Generating plan...")
plan = generate_plan(test_q)
print(f"Plan:\n{plan}")

# Step 2: N answer votes
print(f"\n[2] Generating {CONFIG['n_votes']} answer votes...")
answers = []
for i in range(CONFIG["n_votes"]):
    raw  = generate_answer(test_q, plan)
    pred = extract_pred_answer(raw)
    answers.append(pred)
    print(f"  Vote {i+1}: {pred}")

# Step 3: Vote
print("\n[3] Voting...")
decision = vote_and_decide(answers, test_q, plan)
print(f"  Vote counts  : {decision['vote_counts']}")
print(f"  Strategy     : {decision['strategy']}")
print(f"  Final answer : {decision['final_answer']}")
print(f"  GT answer    : {test_gt}")
print(f"  Correct      : {decision['final_answer'] == test_gt} {'✅' if decision['final_answer'] == test_gt else '❌'}")

print("\nIf this looks correct → run Cell 11 for full evaluation")

SINGLE QUESTION PIPELINE TEST
Question : The girls are trying to raise money for a carnival. Kim raises $320 more than Alexandra, w...
GT Answer: 2280

[1] Generating plan...
Plan:
Step 1: Calculate how much each person raised individually.
Step 2: Sum up their individual amounts to find total fundraising.

[2] Generating 5 answer votes...
  Vote 1: 1880
  Vote 2: 2180
  Vote 3: 2180
  Vote 4: 1880
  Vote 5: 2180

[3] Voting...
  Vote counts  : {'1880': 2, '2180': 3}
  Strategy     : majority
  Final answer : 2180
  GT answer    : 2280
  Correct      : False ❌

If this looks correct → run Cell 11 for full evaluation


In [ ]:
# CELL 11: Full Evaluation Loop
# Expected time: ~3-4 hours for 200 questions (5 votes each)
# Checkpoint every 20 questions — safe to resume

print(f"Evaluating on {len(test_data)} questions...")
print("-" * 55)

all_results = []
start_idx   = 0

if os.path.exists(CONFIG["checkpoint_file"]):
    with open(CONFIG["checkpoint_file"]) as f:
        ckpt = json.load(f)
    start_idx = ckpt.get("last_index", 0)
    if os.path.exists(CONFIG["results_file"]):
        with open(CONFIG["results_file"]) as f:
            all_results = [json.loads(l) for l in f if l.strip()]
    print(f"Resumed: idx={start_idx}, saved={len(all_results)}")
else:
    print("Starting fresh...")

for idx in tqdm(range(start_idx, len(test_data)), desc="Evaluating"):
    item      = test_data[idx]
    question  = item["question"]
    gt_answer = extract_gt_answer(item["answer"])

    try:
        # Guide plan
        plan = generate_plan(question)

        # N votes
        raw_answers = []
        for _ in range(CONFIG["n_votes"]):
            raw  = generate_answer(question, plan)
            pred = extract_pred_answer(raw)
            raw_answers.append(pred)

        # Decide
        decision = vote_and_decide(raw_answers, question, plan)

        all_results.append({
            "question"     : question,
            "gt_answer"    : gt_answer,
            "final_answer" : decision["final_answer"],
            "correct"      : decision["final_answer"] == gt_answer,
            "strategy"     : decision["strategy"],
            "vote_counts"  : decision["vote_counts"],
            "confidence"   : decision["confidence"],
            "plan"         : plan,
        })

    except RuntimeError as e:
        all_results.append({
            "question"     : question,
            "gt_answer"    : gt_answer,
            "final_answer" : "",
            "correct"      : False,
            "strategy"     : "error",
            "error"        : str(e),
        })

    if (idx + 1) % CONFIG["save_every"] == 0:
        with open(CONFIG["results_file"], "w") as f:
            for r in all_results: f.write(json.dumps(r) + "\n")
        with open(CONFIG["checkpoint_file"], "w") as f:
            json.dump({"last_index": idx + 1}, f)

# Final save
with open(CONFIG["results_file"], "w") as f:
    for r in all_results: f.write(json.dumps(r) + "\n")

correct = sum(1 for r in all_results if r["correct"])
print(f"\nDone. Accuracy: {correct}/{len(all_results)} = {correct/len(all_results)*100:.1f}%")

In [ ]:
# CELL 12: Final Report

print("=" * 60)
print("GUIDE PIPELINE — EVALUATION REPORT")
print("=" * 60)

total   = len(all_results)
correct = sum(1 for r in all_results if r["correct"])
accuracy = correct / total * 100

# Strategy breakdown
strategy_counts  = Counter(r["strategy"] for r in all_results)
strategy_correct = {}
for s in strategy_counts:
    subset = [r for r in all_results if r["strategy"] == s]
    strategy_correct[s] = sum(1 for r in subset if r["correct"]) / len(subset) * 100

# Confidence breakdown
high_conf  = [r for r in all_results if r.get("confidence", 0) >= 0.8]
low_conf   = [r for r in all_results if r.get("confidence", 0) < 0.8]
hc_acc     = sum(1 for r in high_conf if r["correct"]) / max(1, len(high_conf)) * 100
lc_acc     = sum(1 for r in low_conf  if r["correct"]) / max(1, len(low_conf))  * 100

print(f"\nOverall Accuracy    : {correct}/{total} = {accuracy:.1f}%")

print(f"\nBy Strategy:")
for s, count in sorted(strategy_counts.items(), key=lambda x: -x[1]):
    acc = strategy_correct[s]
    print(f"  {s:22s}: {count:4d} questions | {acc:.1f}% accuracy")

print(f"\nBy Confidence:")
print(f"  High conf (>=80%) : {len(high_conf):4d} questions | {hc_acc:.1f}% accuracy")
print(f"  Low  conf (< 80%) : {len(low_conf):4d}  questions | {lc_acc:.1f}% accuracy")

# Save
report = {
    "total"            : total,
    "correct"          : correct,
    "accuracy"         : round(accuracy, 2),
    "strategy_counts"  : dict(strategy_counts),
    "strategy_accuracy": {k: round(v, 2) for k, v in strategy_correct.items()},
    "high_conf_acc"    : round(hc_acc, 2),
    "low_conf_acc"     : round(lc_acc, 2),
}
with open(CONFIG["report_file"], "w") as f:
    json.dump(report, f, indent=2)

print(f"\nReport saved: {CONFIG['report_file']}")
print("=" * 60)
print("Pipeline evaluation complete.")
print("Commit this notebook to save results.")
print("=" * 60)